# Patch prediction aggregation


In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
while not (PROJECT_ROOT / "modeling_pipeline.py").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Run this code from the MMDLPC_Code_PDF folder.")
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT.parent / "MMDLPC_Code_PDF_Data"
OUTPUT_DIR = PROJECT_ROOT / '03_Pathomics/00_Preprocessing'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXTERNAL_INPUT_DIR = Path(os.environ.get('MMDLPC_EXTERNAL_INPUTS', str(DATA_ROOT / 'External_Inputs')))


## Patch


In [ ]:
import re

def id_map(x):
    if x.startswith('TCGA') :
        return x[:12] 
    else:
        items = re.split('[ |\-|_]', x)
        return items[0]

In [ ]:
import pandas as pd
from onekey_algo.custom.utils import key2

train_log = pd.read_csv(str(EXTERNAL_INPUT_DIR / '20230206/resnet50/viz/BST_TRAIN_RESULTS.txt'), sep='\t',
                        names=['fname', 'prob', 'pred', 'gt'])

val_log = pd.read_csv(str(EXTERNAL_INPUT_DIR / '20230206/resnet50/viz/BST_VAL_RESULTS.txt'), sep='\t',
                      names=['fname', 'prob', 'pred', 'gt'])
log = pd.concat([train_log, val_log], axis=0)
log['prob'] = log['prob'].round(decimals=2)
log[['group']] = log[['fname']].applymap(id_map)
log

In [ ]:
import os

os.makedirs(str(OUTPUT_DIR), exist_ok=True)
results = key2.key2histogram(log, group_column='group',histo_columns='prob', norm=True)
results.to_csv(str(OUTPUT_DIR / 'Patch_superwise_path_prob_histogram.csv'), header=True, index=False)
display(results)

results = key2.key2histogram(log, group_column='group',histo_columns='pred', norm=True)
results.to_csv(str(OUTPUT_DIR / 'Patch_superwise_path_pred_histogram.csv'), header=True, index=False)
display(results)

In [ ]:
results = key2.key2tfidf(log, group_column='group',corpus_columns='prob')
results.to_csv(str(OUTPUT_DIR / 'Patch_superwise_path_prob_tfidf.csv'), header=True, index=False)
display(results)

results = key2.key2tfidf(log, group_column='group',corpus_columns='pred')
results.to_csv(str(OUTPUT_DIR / 'Patch_superwise_path_pred_tfidf.csv'), header=True, index=False)
display(results)